In [1]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

# Mining Curation

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Curation.ipynb)

Perform deterministic clustering, diverse sample selection, and class imbalance mitigation.

## Jump to an API

- [mining.cluster](#mining.cluster)
- [mining.select](#mining.select)
- [mining.balance](#mining.balance)
- [mining.ClusterResult](#mining.ClusterResult)

### mining.cluster

Group images using histograms, HOG, quality, boxes, and class presence.

In [2]:

# Load a materialized YOLO dataset
from klygo import archive, datasets, files, mining

# Download and convert COCO8 into a normalized flat YOLO dataset
workspace = files.join("artifacts", "tutorials", "mining")
flat_dir = files.join(workspace, "flat_coco8")
if not files.exists(files.join(flat_dir, "data.yaml")):
    files.mkdir(workspace)
    zip_path = files.download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", workspace, overwrite=True, verbose=False)
    yaml_path = files.download("https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco8.yaml", workspace, overwrite=True, verbose=False)
    raw_dir = files.join(workspace, "raw_coco8")
    archive.extract(zip_path, raw_dir, overwrite=True, verbose=False)
    files.copy(yaml_path, files.join(raw_dir, "coco8.yaml"), overwrite=True)
    datasets.convert(
        source=raw_dir,
        target=flat_dir,
        tree={
            "images": ["coco8/images/train", "coco8/images/val"],
            "labels": ["coco8/labels/train", "coco8/labels/val"],
            "metadata": "coco8.yaml",
        },
        on_conflict="rename",
        overwrite=True,
    )

data = datasets.load(flat_dir, format="yolo", verbose=False)
print(f"Loaded dataset: {len(data)} images, {data.total_objects} boxes, classes={list(data.names.values())}")

# Cluster dataset into visual groups
clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
clusters.show()
print("Cluster representatives:", clusters.representatives)

Loaded dataset: 8 images, 30 boxes, classes=['bowl', 'broccoli', 'orange', 'giraffe', 'potted plant', 'vase', 'zebra', 'umbrella', 'person', 'dog', 'horse', 'elephant']
           ClusterResult           
┌─────────┬──────┬────────────────┐
│ Cluster │ Size │ Representative │
├─────────┼──────┼────────────────┤
│ 0       │ 4    │ 4              │
│ 1       │ 3    │ 3              │
│ 2       │ 1    │ 0              │
└─────────┴──────┴────────────────┘
Cluster representatives: {0: 4, 1: 3, 2: 0}


### mining.select

Select whole images by class, quality, boxes, random sampling, quota, rarity, diversity, or score.

In [3]:
# Select using random sampling
sel_rand = mining.select(data, method="random", limit=4, seed=42)
print(f"Random selection: {len(sel_rand)} images")

# Select using visual cluster diversity
sel_div = mining.select(data, method="diversity", cluster_result=clusters, limit=3)
print(f"Diversity selection: {len(sel_div)} images")

# Select using multi-criteria weighted scoring
scores = mining.score(data)
sel_weight = mining.select(data, method="weighted", scores=scores, limit=4)
print(f"Weighted selection: {len(sel_weight)} images")

Random selection: 4 images
Diversity selection: 3 images
Weighted selection: 4 images


### mining.balance

Reduce imbalance without creating duplicate frame identities.

In [4]:
# Balance dataset targeting median class count
balanced = mining.balance(data, target="median", count_by="images", seed=42)
print(f"Original {len(data)} -> Balanced {len(balanced)} images")

Original 8 -> Balanced 8 images


### mining.ClusterResult

Cluster assignments, groups, and representatives returned by `cluster`.

In [5]:
clusters.save("artifacts/clusters.json")
print("Saved ClusterResult to JSON.")

Saved ClusterResult to JSON.


## Put it together

Cluster images by visual features, select diverse representatives, and balance class distributions.

In [6]:
clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
curated = mining.select(data, method="diversity", cluster_result=clusters, limit=3)
balanced = mining.balance(curated, target="median", seed=42)
balanced.export("artifacts/curated_balanced", format="yolo", overwrite=True)
print("Curated and balanced dataset exported.")

Curated and balanced dataset exported.


## Remember

1. `cluster` uses deterministic K-Means over normalized visual and annotation features.
2. `select` supports 9 curation strategies: random, weighted, quality, rare_first, diversity, boxes, quota, coverage, and class.
3. `balance` reduces class imbalance without synthesizing or duplicating images.